# Task 2.3 — Results Comparison, Confusion Matrix & Reproducibility Checklist

In [1]:
# ── Imports & seed ────────────────────────────────────────────────
import os, numpy as np, random
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

C_VALUE, MAX_ITER = 1.0, 5000
RESULTS_DIR = "/Users/abhijeetkrshah/Desktop/230036-midsem/partB/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

wine = load_wine()
X, y = wine.data, wine.target
X_sc = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

model = LinearSVC(multi_class='crammer_singer', C=C_VALUE,
                  max_iter=MAX_ITER, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
test_err = (1 - accuracy_score(y_test, y_pred)) * 100
print(f"Wine CS test error: {test_err:.2f}%")


Wine CS test error: 5.56%


## Results vs Paper (Table 2)

| Setting | Test Error (%) |
|---------|---------------|
| **Ours** — CS / Wine / sklearn LinearSVC | *(see output above)* |
| **Paper** — CS / CB513 / MSVMpack | **23.63%** |

**Explaining the gap (3–5 sentences):**  
The Wine dataset contains 178 well-separated samples across 3 classes, whereas CB513 has 84,119 protein sequences with naturally overlapping secondary-structure labels — dataset difficulty alone accounts for most of the gap.  
Sklearn's `LinearSVC` uses a coordinate-descent solver (liblinear), not the Frank-Wolfe + LP decomposition used by MSVMpack; both converge to the same global solution for convex problems, but the approximation path differs.  
MSVMpack's CB513 experiments use profile-based string kernels tuned to protein sequence data, while we use a simple linear kernel on standardised physicochemical features — very different kernel inductive biases.  
We fix $C=1.0$ without cross-validation; the paper performs a full CV grid search over $C$ and kernel parameters, which can recover several percentage points of error.  
Finally, 178 training samples give very low statistical uncertainty; CB513's much larger test set produces a more stable and harder error estimate.

In [2]:
# ── Confusion matrix ──────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=wine.target_names,
            yticklabels=wine.target_names,
            linewidths=0.5, ax=ax)
ax.set_title(f"Confusion Matrix — CS M-SVM on Wine\nTest Error: {test_err:.2f}%  |  Paper CB513 CS: 23.63%", pad=10)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
path = os.path.join(RESULTS_DIR, "confusion_matrix.png")
plt.savefig(path, dpi=150); plt.show()
print(f"Saved → {path}")


Saved → /Users/abhijeetkrshah/Desktop/230036-midsem/partB/results/confusion_matrix.png


/var/folders/t3/st41wyns54d8gvsrqb52wvgh0000gn/T/ipykernel_4851/579000336.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(path, dpi=150); plt.show()


## Reproducibility Checklist

| Item | Status |
|------|--------|
| `random_state=42` set at top of every notebook | ✅ |
| `requirements.txt` lists exact versions (`numpy==1.26.4`, `scikit-learn==1.4.2`, `matplotlib==3.8.4`, `seaborn==0.13.2`, `scipy==1.13.0`) | ✅ |
| All notebooks execute top-to-bottom without errors | ✅ Verified via nbconvert |
| No manual dataset steps — `load_wine()` is automatic | ✅ |
| All hyperparameters (`C_VALUE`, `MAX_ITER`, `RANDOM_STATE`) defined at top of each notebook | ✅ |
| All plots saved to `partB/results/` via `plt.savefig()` | ✅ |
| No GPU code — CPU only | ✅ |